# Predicción de hits musicales

Este notebook documenta el análisis del dataset y ejecuta el entrenamiento reproducible definido en [`train.py`](../train.py). El objetivo binario es `hit = 1` cuando `Popularity >= 70`; de lo contrario, `hit = 0`.

El archivo `data/train.csv` se usa para aprender y evaluar. `data/test.csv` se inspecciona para entender su esquema, pero no se usa como conjunto de evaluación: se reserva un test interno por artista para poder medir contra etiquetas conocidas. El artefacto final se guarda en `models/best_model.joblib`.

## 1. Carga de datos y configuración

La búsqueda de la raíz hace que las rutas funcionen tanto si Jupyter se inicia desde la raíz del proyecto como desde `notebooks/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "train.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró train.py; abre el notebook dentro del proyecto.")

TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print(f"Raíz del proyecto: {ROOT}")
print(f"Train: {train_df.shape[0]:,} filas x {train_df.shape[1]} columnas")
print(f"Test:  {test_df.shape[0]:,} filas x {test_df.shape[1]} columnas")
display(train_df.head())

## 2. Esquema, faltantes, duplicados y estadísticas

`Popularity` es la variable objetivo. Las 428 filas sin Popularity no sirven para aprendizaje supervisado y se eliminan antes de entrenar. Los faltantes de las características se imputan dentro del pipeline, para que las estadísticas de imputación se aprendan solo del conjunto de ajuste. `describe(include="all")` resume distribuciones numéricas y categorías de texto.

In [ ]:
def dataset_profile(frame, name):
    print(f"\n{name}: {frame.shape[0]:,} filas x {frame.shape[1]} columnas")
    profile = pd.DataFrame({
        "tipo": frame.dtypes.astype(str),
        "nulos": frame.isna().sum(),
        "% nulos": (frame.isna().mean() * 100).round(2),
        "valores_unicos": frame.nunique(dropna=True),
    })
    display(profile)
    print(f"Filas completamente duplicadas: {frame.duplicated().sum():,}")
    display(frame.describe(include="all").transpose())

dataset_profile(train_df, "Train")
dataset_profile(test_df, "Test (solo inspección)")

## 3. Variable objetivo y proporción de hits

Se transforma Popularity a una etiqueta binaria con el corte solicitado (70). Esta variable está desbalanceada: por eso Accuracy por sí sola puede resultar engañosa. F1 evalúa el equilibrio entre precisión y recall de la clase hit, mientras AUC-ROC mide la capacidad de ordenar positivos por encima de negativos.

In [ ]:
TARGET = "Popularity"
THRESHOLD = 70
target_data = train_df[TARGET].dropna()
hit_counts = target_data.ge(THRESHOLD).map({True: "hit (>=70)", False: "no hit (<70)"}).value_counts()
target_summary = pd.DataFrame({"canciones": hit_counts, "proporción": hit_counts / len(target_data)})
display(target_summary)
print(f"Popularity: media={target_data.mean():.2f}, mediana={target_data.median():.0f}, "
      f"desviación estándar={target_data.std():.2f}, mínimo={target_data.min():.0f}, máximo={target_data.max():.0f}")
print(f"Filas sin Popularity: {train_df[TARGET].isna().sum():,}")

## 4. Selección de predictores y prevención de fuga

Se usan las 13 medidas de audio indicadas abajo: describen características sonoras disponibles al analizar una canción y se encuentran tanto en train como en test. `Artist Name` solo sirve para agrupar la partición; nunca es entrada del modelo. `Track Name` identifica la canción, y `Class` es una etiqueta auxiliar no disponible en test, así que también se excluye. Se omiten asimismo IDs, streams, playlists y rankings: pueden revelar directa o indirectamente popularidad y causar fuga de información.

Los tipos de estas 13 variables en el archivo son numéricos. El script incluye imputación mediana para variables numéricas y one-hot para cualquier predictor categórico que se añada posteriormente.

In [ ]:
FEATURES = [
    "danceability", "energy", "key", "loudness", "mode", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
    "duration_in min/ms", "time_signature",
]
excluded = ["Artist Name (solo grupos)", "Track Name", "Class", "Popularity (objetivo)"]
feature_meanings = {
    "danceability": "Qué tan bailable es",
    "energy": "Intensidad y actividad percibidas",
    "key": "Tonalidad musical estimada",
    "loudness": "Volumen medio, en dB",
    "mode": "Modalidad mayor/menor",
    "speechiness": "Presencia de palabras habladas",
    "acousticness": "Probabilidad de sonido acústico",
    "instrumentalness": "Probabilidad de que no tenga voz",
    "liveness": "Probabilidad de grabación en vivo",
    "valence": "Positividad musical percibida",
    "tempo": "Velocidad musical estimada",
    "duration_in min/ms": "Duración; nombre sugiere unidades mixtas",
    "time_signature": "Compás estimado",
}
selection = pd.DataFrame({"decisión": "Usar como predictor", "significado": feature_meanings}, index=FEATURES)
display(selection)
print("Excluidas:", ", ".join(excluded))
missing_features = sorted(set(FEATURES) - set(train_df.columns))
if missing_features:
    raise ValueError(f"Faltan predictores en train.csv: {missing_features}")
if not set(FEATURES).issubset(test_df.columns):
    raise ValueError("El conjunto de predictores no coincide con test.csv")

## 5. Método de entrenamiento y evaluación

El script limpia duplicados y etiquetas faltantes, y separa el 20 % de los datos por `Artist Name` (`GroupShuffleSplit`): así, ningún artista aparece a la vez en train y test. Dentro del train reserva otra validación por artista para elegir el umbral que maximiza F1; el test final no se usa para esa elección. Compara un baseline de clase mayoritaria (`DummyClassifier`) y RandomForest con pesos equilibrados. Reporta F1, AUC-ROC, Accuracy y matrices de confusión. La importancia por permutación mide cuánto cae F1 al mezclar cada característica del test.

El modelo escogido y el umbral aprendido se guardan juntos en el archivo joblib para que el criterio de clasificación pueda reproducirse.

In [ ]:
import subprocess

MODEL_PATH = ROOT / "models" / "best_model.joblib"
result = subprocess.run(
    [sys.executable, str(ROOT / "train.py"), "--data", str(TRAIN_PATH), "--model-out", str(MODEL_PATH)],
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)
if result.stderr:
    print("Mensajes del proceso:")
    print(result.stderr)

## 6. Interpretación y límites

En una ejecución de referencia sobre este dataset, el conjunto de train contiene 17.568 filas tras retirar los objetivos faltantes y solo el 8,8 % son hits. Con una separación aleatoria por artista no hubo artistas compartidos; RandomForest obtuvo F1 ≈ 0,31 y AUC-ROC ≈ 0,75, por encima del baseline. La matriz de confusión y las importancias impresas arriba se recalculan al ejecutar el notebook; variarán si cambian los datos, el código o las versiones.

En esa evaluación, `danceability`, `loudness`, `duration_in min/ms`, `instrumentalness` y `acousticness` estuvieron entre las variables más importantes por permutación. Esto describe asociación predictiva, no causalidad. `duration_in min/ms` tiene valores de escalas muy distintas (mínimo cercano a 0,5 y mediana cercana a 209.160), por lo que su unidad/formato merece revisión. El dataset también está desbalanceado y el umbral se ajusta para F1; estas métricas no garantizan resultados equivalentes en producción.

## 7. Ejecución fuera del notebook

Desde la raíz del repositorio en Windows:

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
.\.venv\Scripts\python.exe train.py
```

Para especificar rutas: `python train.py --data data/train.csv --model-out models/best_model.joblib`. El artefacto guarda `model`, `features`, el corte de popularity y el umbral de probabilidad aprendido.